# Ribosome-Cascade: Scale-Up Experiments (Colab A100)
## Validating the "lighter model on metatokens" thesis at 250M+ params

**What we proved locally:**
A 49M ribosome-tiny model (2 embed + 4 upper layers, 16 metatokens) achieves PPL 4.12 vs a 63M standard transformer's PPL 1063 on OpenWebText at 100K steps.

**What this notebook tests:**
1. Does the advantage hold at 250M params? (hidden=1024, more layers)
2. Does lower CE translate to downstream task accuracy? (HellaSwag, LAMBADA)


In [ ]:
# Setup
!pip install -q transformers accelerate

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os, sys, json, time, math

print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")


In [ ]:
# Mount Google Drive EARLY so results are saved even if session disconnects
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted - results will auto-save to Drive')


In [ ]:
# Clone repo for architecture code
import os
if os.path.exists('Ribosome-Cascade'):
    !cd Ribosome-Cascade && git pull
else:
    !git clone https://github.com/J-Lehrer/Ribosome-Cascade.git
%cd Ribosome-Cascade

sys.path.insert(0, "/content/Ribosome-Cascade")
from native_arch_v1 import RMSNorm, RotaryEmbedding, TransformerBlock, RibosomeLayer, CascadeProcessor, ChunkDecoder
print("Architecture loaded")


## 1. Download training data (no HuggingFace)

In [ ]:
# Download wikitext-103 directly from source (fast, reliable)
import urllib.request, zipfile, os

DATA_DIR = '/content/wikitext-103-raw'
ZIP_PATH = '/content/wikitext-103-raw-v1.zip'
URL = "https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-103-raw-v1.zip"

if not os.path.exists(f'{DATA_DIR}/wiki.train.raw'):
    print("Downloading wikitext-103...")

    def _progress(count, block_size, total_size):
        pct = count * block_size * 100 / total_size
        mb = count * block_size / 1e6
        total_mb = total_size / 1e6
        print(f"\r  {pct:.0f}% ({mb:.0f}/{total_mb:.0f} MB)", end="", flush=True)

    urllib.request.urlretrieve(URL, ZIP_PATH, reporthook=_progress)
    print("\nUnzipping...")

    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')

    if os.path.exists('/content/wikitext-103-raw-v1'):
        !mv /content/wikitext-103-raw-v1 {DATA_DIR}
    !ls -lh {DATA_DIR}/
    os.remove(ZIP_PATH)
    print("Done")
else:
    print(f"Already have {DATA_DIR}/wiki.train.raw")


In [ ]:
# Tokenize and cache (runs once, ~30s)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_file(path, cache_path):
    if os.path.exists(cache_path):
        print(f"  Loading cached: {cache_path}")
        return torch.load(cache_path, weights_only=True).tolist()
    print(f"  Tokenizing: {path}")
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    ids = tokenizer.encode(text)
    torch.save(torch.tensor(ids, dtype=torch.int32), cache_path)
    print(f"  -> {len(ids):,} tokens cached to {cache_path}")
    return ids

train_ids = tokenize_file(f'{DATA_DIR}/wiki.train.raw', '/content/train_tokens.pt')
val_ids = tokenize_file(f'{DATA_DIR}/wiki.valid.raw', '/content/val_tokens.pt')
print(f"\nTrain: {len(train_ids):,} tokens  Val: {len(val_ids):,} tokens")


## 2. Architecture at 250M scale

In [ ]:
# Model definitions

class BigBaseline(nn.Module):
    """Standard 16-layer transformer on raw tokens."""
    def __init__(self, vocab_size=50257, hidden_size=1024, n_heads=16,
                 n_layers=16, max_seq_len=1024):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, hidden_size)
        self.rope = RotaryEmbedding(hidden_size // n_heads, max_seq_len)
        self.layers = nn.ModuleList([
            TransformerBlock(hidden_size, n_heads, self.rope) for _ in range(n_layers)
        ])
        self.norm = RMSNorm(hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: torch.nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                torch.nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, labels=None):
        B, S = input_ids.shape
        x = self.tok_emb(input_ids)
        causal = torch.triu(torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1)
        for layer in self.layers:
            x = layer(x, causal_mask=causal)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            # No shift: data loader already provides aligned input/label pairs
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))
        return loss, logits

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


class RibosomeTiny(nn.Module):
    """Ribosome compresses 1024->64 tokens, then 6-layer upper transformer on metatokens."""
    def __init__(self, vocab_size=50257, hidden_size=1024, n_heads=16,
                 embed_layers=4, upper_layers=6, max_seq_len=1024, n_chunks=64):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, hidden_size)
        self.rope = RotaryEmbedding(hidden_size // n_heads, max_seq_len)
        self.embed = nn.ModuleList([
            TransformerBlock(hidden_size, n_heads, self.rope) for _ in range(embed_layers)
        ])
        self.embed_norm = RMSNorm(hidden_size)
        self.ribosome = RibosomeLayer(hidden_size, n_chunks, n_heads)
        self.upper = nn.ModuleList([
            TransformerBlock(hidden_size, n_heads) for _ in range(upper_layers)
        ])
        self.upper_norm = RMSNorm(hidden_size)
        self.decoder = ChunkDecoder(hidden_size, n_heads)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        self.ribosome_alpha = 1.0
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: torch.nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                torch.nn.init.normal_(m.weight, std=0.02)

    def forward(self, input_ids, labels=None):
        B, S = input_ids.shape
        x = self.tok_emb(input_ids)
        causal = torch.triu(torch.ones(S, S, device=x.device, dtype=torch.bool), diagonal=1)
        for layer in self.embed:
            x = layer(x, causal_mask=causal)
        token_states = self.embed_norm(x)
        chunk_repr, chunk_weights, assign, importance = self.ribosome(token_states)
        for layer in self.upper:
            chunk_repr = layer(chunk_repr)
        chunk_repr = self.upper_norm(chunk_repr)
        decoded = self.decoder(token_states, chunk_repr, assign)
        output = self.ribosome_alpha * decoded + (1 - self.ribosome_alpha) * token_states
        logits = self.lm_head(output)
        loss = None
        if labels is not None:
            # No shift: data loader already provides aligned input/label pairs
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1))
        return loss, logits, importance

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


# Verify
V = len(tokenizer)
big = BigBaseline(vocab_size=V); tiny = RibosomeTiny(vocab_size=V)
print(f"Big baseline: {big.count_params():,} params")
print(f"Ribosome tiny: {tiny.count_params():,} params")
del big, tiny


## 3. Training infrastructure

In [ ]:
# Data loader + training loop (all local, no network calls)

from torch.utils.data import Dataset, DataLoader

class TokenDataset(Dataset):
    def __init__(self, token_ids, max_length):
        self.ids = token_ids
        self.max_length = max_length
        self.n_chunks = (len(token_ids) - 1) // max_length

    def __len__(self):
        return self.n_chunks

    def __getitem__(self, idx):
        start = idx * self.max_length
        chunk = self.ids[start:start + self.max_length + 1]
        return {"input_ids": torch.tensor(chunk[:-1], dtype=torch.long),
                "labels": torch.tensor(chunk[1:], dtype=torch.long)}


def get_lr(step, total_steps, max_lr, min_lr, warmup_steps):
    if step < warmup_steps:
        return max_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


def train_model(model, name, train_loader, val_loader, device, config, is_ribosome=False):
    total_steps = config["total_steps"]
    grad_accum = config["grad_accum"]
    max_lr = config["max_lr"]
    min_lr = config["min_lr"]
    warmup_steps = int(total_steps * 0.05)
    alpha_ramp_steps = int(total_steps * 0.10)
    log_every = config.get("log_every", 200)
    eval_every = config.get("eval_every", 5000)
    out_dir = config["out_dir"]
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"Params: {model.count_params():,}")
    print(f"Steps: {total_steps}, warmup: {warmup_steps}")
    print(f"{'='*60}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, betas=(0.9, 0.95), weight_decay=0.1)
    model.train()
    global_step = 0
    best_val_loss = float("inf")
    log_history = []
    epoch = 0
    t0 = time.time()

    while global_step < total_steps:
        epoch += 1
        epoch_losses = []
        optimizer.zero_grad()

        for batch_idx, batch in enumerate(train_loader):
            if global_step >= total_steps:
                break
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            lr = get_lr(global_step, total_steps, max_lr, min_lr, warmup_steps)
            for pg in optimizer.param_groups:
                pg["lr"] = lr

            if is_ribosome:
                if global_step < alpha_ramp_steps:
                    model.ribosome_alpha = global_step / alpha_ramp_steps
                else:
                    model.ribosome_alpha = 1.0
                model.ribosome.gumbel_temperature = 1.0 - 0.9 * min(global_step / max(total_steps, 1), 1.0)
                loss, logits, importance = model(input_ids, labels)
            else:
                loss, logits = model(input_ids, labels)

            loss = loss / grad_accum
            loss.backward()
            epoch_losses.append(loss.item() * grad_accum)

            if (batch_idx + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % log_every == 0:
                    mean_loss = np.mean(epoch_losses[-log_every * grad_accum:])
                    elapsed = time.time() - t0
                    sps = global_step / elapsed
                    eta = (total_steps - global_step) / sps / 3600 if sps > 0 else 0
                    print(f"  step={global_step:>6d}  CE={mean_loss:.4f}  lr={lr:.2e}  "
                          f"ep={epoch}  {sps:.1f}s/s  ETA={eta:.1f}h")
                    log_history.append({"step": global_step, "train_loss": float(mean_loss), "lr": lr})
                    with open(os.path.join(out_dir, "training_log.json"), "w") as f:
                        json.dump(log_history, f)

                if global_step % eval_every == 0:
                    model.eval()
                    val_losses = []
                    with torch.no_grad():
                        for vb in val_loader:
                            vi, vl = vb["input_ids"].to(device), vb["labels"].to(device)
                            if is_ribosome:
                                vv, _, _ = model(vi, vl)
                            else:
                                vv, _ = model(vi, vl)
                            val_losses.append(vv.item())
                    val_loss = np.mean(val_losses)
                    ppl = np.exp(val_loss)
                    print(f"  >>> VAL={val_loss:.4f} PPL={ppl:.1f} (best={best_val_loss:.4f})")
                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        torch.save({"step": global_step, "model": model.state_dict(),
                                    "val_loss": val_loss, "params": model.count_params()},
                                   os.path.join(out_dir, "best.pt"))
                    model.train()

    torch.save({"step": global_step, "model": model.state_dict(),
                "val_loss": best_val_loss, "params": model.count_params()},
               os.path.join(out_dir, "final.pt"))
    with open(os.path.join(out_dir, "training_log.json"), "w") as f:
        json.dump(log_history, f, indent=2)

    elapsed = time.time() - t0
    print(f"\n  Done: {name} - best val CE={best_val_loss:.4f} PPL={np.exp(best_val_loss):.1f} ({elapsed/3600:.1f}h)")
    return best_val_loss


MAX_LENGTH = 1024
BATCH_SIZE = 4
GRAD_ACCUM = 8

train_dataset = TokenDataset(train_ids, MAX_LENGTH)
val_dataset = TokenDataset(val_ids, MAX_LENGTH)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

steps_per_epoch = len(train_loader) // GRAD_ACCUM
EPOCHS = 16
TOTAL_STEPS = steps_per_epoch * EPOCHS

print(f"Train: {len(train_dataset):,} chunks, {steps_per_epoch:,} steps/epoch")
print(f"Val: {len(val_dataset):,} chunks")
print(f"Total: {TOTAL_STEPS:,} steps ({EPOCHS} epochs)")

config = {
    "total_steps": TOTAL_STEPS, "grad_accum": GRAD_ACCUM,
    "max_lr": 3e-4, "min_lr": 3e-5, "log_every": 200, "eval_every": 5000,
}

device = torch.device("cuda")
total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
frac = min(35.0 / total_vram, 0.95)
torch.cuda.set_per_process_memory_fraction(frac)
print(f"VRAM cap: {35.0}GB ({frac:.0%} of {total_vram:.0f}GB)")


## 4. Train both models

In [ ]:
# Train big baseline
big = BigBaseline(vocab_size=V, hidden_size=1024, n_heads=16,
                  n_layers=16, max_seq_len=MAX_LENGTH).to(device)

config["out_dir"] = "/content/scale_experiment/big_16L_1024h"
big_val = train_model(big, "big_16L_1024h", train_loader, val_loader, device, config)
print(f"\nBig baseline final val CE: {big_val:.4f}")
del big; torch.cuda.empty_cache()

# Auto-save big baseline to Drive immediately
import shutil
_drv = '/content/drive/MyDrive/Ribosome-Cascade/scale_experiment_corrected/big_16L_1024h'
_loc = '/content/scale_experiment/big_16L_1024h'
os.makedirs(_drv, exist_ok=True)
for _f in os.listdir(_loc):
    shutil.copy2(os.path.join(_loc, _f), os.path.join(_drv, _f))
print(f"Big baseline saved to Drive ({_drv})")


In [ ]:
# Train ribosome tiny
tiny = RibosomeTiny(vocab_size=V, hidden_size=1024, n_heads=16,
                    embed_layers=4, upper_layers=6,
                    max_seq_len=MAX_LENGTH, n_chunks=64).to(device)

config["out_dir"] = "/content/scale_experiment/ribosome_tiny_1024h"
tiny_val = train_model(tiny, "ribosome_tiny_1024h", train_loader, val_loader, device, config,
                        is_ribosome=True)
print(f"\nRibosome tiny final val CE: {tiny_val:.4f}")
del tiny; torch.cuda.empty_cache()

# Auto-save ribosome to Drive immediately
_drv = '/content/drive/MyDrive/Ribosome-Cascade/scale_experiment_corrected/ribosome_tiny_1024h'
_loc = '/content/scale_experiment/ribosome_tiny_1024h'
os.makedirs(_drv, exist_ok=True)
for _f in os.listdir(_loc):
    shutil.copy2(os.path.join(_loc, _f), os.path.join(_drv, _f))
print(f"Ribosome tiny saved to Drive ({_drv})")


In [ ]:
# Push checkpoints to Google Drive (already mounted at top)
import shutil

DRIVE_DIR = '/content/drive/MyDrive/Ribosome-Cascade/scale_experiment_corrected'
LOCAL_DIR = '/content/scale_experiment'
os.makedirs(DRIVE_DIR, exist_ok=True)

for root, dirs, files in os.walk(LOCAL_DIR):
    for f in files:
        src = os.path.join(root, f)
        rel = os.path.relpath(src, LOCAL_DIR)
        dst = os.path.join(DRIVE_DIR, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print(f'  copied: {rel} ({os.path.getsize(src)/1e6:.1f} MB)')

print(f'\nDone - pushed to {DRIVE_DIR}')


## 5. Results

In [ ]:
# Head-to-head comparison
print("=" * 60)
print("250M SCALE RESULTS")
print("=" * 60)
print(f"Big baseline (16L, 1024 tokens): val CE = {big_val:.4f}  PPL = {math.exp(big_val):.0f}")
print(f"Ribosome tiny (10L, 64 chunks):  val CE = {tiny_val:.4f}  PPL = {math.exp(tiny_val):.0f}")
print(f"Delta: {tiny_val - big_val:+.4f} CE")
if big_val > tiny_val:
    print(f"Perplexity ratio: {math.exp(big_val) / math.exp(tiny_val):.1f}x")

results = {"big_val_ce": big_val, "tiny_val_ce": tiny_val}
with open("/content/scale_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved to /content/scale_results.json")
